# Tests de mini-funcionalidades de OP-06 `write_trips`

Este notebook se usa para probar helpers y bloques internos de `write_trips()` antes de hacer tests integrados de la función pública completa.

Objetivo:
- verificar minifuncionalidades de escritura de forma aislada;
- cubrir el contrato actual backend-aware de persistencia de trips;
- probar helpers vigentes para Parquet y Feather;
- dejar una base fácil de portar después a pytest.

Convenciones:
- los tests usan `assert`;
- cuando una prueba necesita inspección visual, se acompaña con `display(...)`;
- las pruebas de este notebook no reemplazan tests integrados;
- este notebook cubre solo OP-06 `write_trips`, no OP-07 `read_trips`.

## Bloque 0. Preparación

### 0.1 Imports generales

Qué prepara: imports básicos, utilidades de filesystem, JSON y dependencias Arrow necesarias para verificar escritura Parquet/Feather.

In [2]:
import copy
import json
import shutil
import tempfile
from pathlib import Path

import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.feather as feather
from pyarrow import types as patypes

### 0.2 Imports del módulo

Qué prepara: imports de clases y helpers reales usados por OP-06 `write_trips`.

Importante: no se importan helpers de `read_trips`, porque este notebook debe quedar separado de OP-07.

In [3]:
from pylondrina.schema import (
    DomainSpec,
    FieldSpec,
    TripSchema,
    TripSchemaEffective,
)

from pylondrina.datasets import TripDataset
from pylondrina.reports import Issue
from pylondrina.errors import ExportError, ValidationError

from pylondrina.io.trips import (
    WriteTripsOptions,
    _validate_write_contract,
    _resolve_write_identity_and_sidecar,
    _create_trips_staging_dir,
    _collect_arrow_categorical_fields,
    _prepare_trips_df_for_arrow_write,
    _write_trips_table_to_staging,
    _write_sidecar_json,
    _assert_staging_complete,
    _commit_staged_trips_artifact,
    _cleanup_staging_dir,
    _build_write_trips_summary,
    _has_golondrina_artifact_suffix,
    _append_golondrina_artifact_suffix,
    _normalize_trips_artifact_root_for_write,
    _trip_data_filename_for_storage,
    _build_storage_options_snapshot,
    _resolve_trips_artifact_paths,
    _assert_json_safe,
    _trip_schema_to_snapshot,
    _trip_schema_effective_to_snapshot,
    _build_issues_summary,
    _build_io_event,
    _append_event,
    _options_to_write_parameters,
    _extract_validated_flag,
)

In [4]:
def show_ok(label: str):
    print(f"OK - {label}")


def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj, ensure_ascii=False)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e


def get_issue_codes(issues):
    return [i.code if hasattr(i, "code") else i.get("code") for i in issues]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró el issue {code}. Codes actuales: {codes}"


def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente el issue {code}. Codes actuales: {codes}"


def assert_counts_by_level(issues, *, errors=None, warnings=None, info=None):
    counts = {"error": 0, "warning": 0, "info": 0}
    for issue in issues:
        counts[issue.level] = counts.get(issue.level, 0) + 1

    if errors is not None:
        assert counts["error"] == errors, f"errors esperado={errors}, actual={counts['error']}"
    if warnings is not None:
        assert counts["warning"] == warnings, f"warnings esperado={warnings}, actual={counts['warning']}"
    if info is not None:
        assert counts["info"] == info, f"info esperado={info}, actual={counts['info']}"


### 0.4 Configuración visual

Qué prepara: display más cómodo para reportes, issues y tablas pequeñas.

In [5]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Imports OK")
show_ok("Sección 0 cargada")

Imports OK
OK - Sección 0 cargada


## Bloque 1. Fixtures reutilizables mínimas

Qué prepara: factories pequeñas para schema, schema_effective, dataframe, TripDataset y payloads de sidecar usados por helpers de escritura.

In [6]:
def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


def make_trip_schema(fields: list[FieldSpec], *, version: str = "1.1") -> TripSchema:
    return TripSchema(
        version=version,
        fields={f.name: f for f in fields},
        required=[f.name for f in fields if f.required],
        semantic_rules=None,
    )


def make_trip_schema_effective(
    *,
    dtype_effective: dict | None = None,
    overrides: dict | None = None,
    domains_effective: dict | None = None,
    temporal: dict | None = None,
    fields_effective: list | None = None,
) -> TripSchemaEffective:
    return TripSchemaEffective(
        dtype_effective=dtype_effective or {},
        overrides=overrides or {},
        domains_effective=domains_effective or {},
        temporal=temporal or {},
        fields_effective=fields_effective or [],
    )


def make_trip_df() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "movement_id": ["m1", "m2", "m3"],
            "trip_id": ["t1", "t2", "t3"],
            "movement_seq": [0, 0, 0],
            "user_id": ["u1", "u2", "u3"],
            "origin_latitude": [-33.45, -33.46, -33.47],
            "origin_longitude": [-70.66, -70.67, -70.68],
            "destination_latitude": [-33.41, -33.42, -33.43],
            "destination_longitude": [-70.61, -70.62, -70.63],
            "mode": ["bus", "metro", "bus"],
            "purpose": ["work", "study", "work"],
            "comment": ["a", "b", "c"],
            "trip_weight": [1.0, 2.5, 1.2],
        }
    )


def make_trip_schema_minimal() -> TripSchema:
    return make_trip_schema(
        [
            make_field("movement_id", "string", required=True),
            make_field("trip_id", "string", required=True),
            make_field("movement_seq", "int", required=True),
            make_field("user_id", "string", required=True),
            make_field("origin_latitude", "float", required=True),
            make_field("origin_longitude", "float", required=True),
            make_field("destination_latitude", "float", required=True),
            make_field("destination_longitude", "float", required=True),
            make_field(
                "mode",
                "categorical",
                required=False,
                domain=DomainSpec(values=["bus", "metro", "walk", "car"], extendable=True),
            ),
            make_field(
                "purpose",
                "categorical",
                required=False,
                domain=DomainSpec(values=["work", "study", "health"], extendable=True),
            ),
            make_field("comment", "string", required=False),
            make_field("trip_weight", "float", required=False),
        ]
    )


def make_trip_schema_effective_minimal() -> TripSchemaEffective:
    return make_trip_schema_effective(
        dtype_effective={
            "mode": "categorical",
            "purpose": "categorical",
            "trip_weight": "float",
        },
        domains_effective={
            "mode": {"values": ["bus", "metro", "walk", "car"]},
            "purpose": {"values": ["work", "study", "health"]},
        },
        temporal={"tier": "tier_3"},
        fields_effective=[
            "movement_id",
            "trip_id",
            "movement_seq",
            "user_id",
            "origin_latitude",
            "origin_longitude",
            "destination_latitude",
            "destination_longitude",
            "mode",
            "purpose",
            "comment",
            "trip_weight",
        ],
    )


def make_tripdataset(
    *,
    validated: bool = True,
    include_dataset_id: bool = True,
    include_artifact_id: bool = False,
) -> TripDataset:
    schema = make_trip_schema_minimal()
    schema_effective = make_trip_schema_effective_minimal()

    metadata = {
        "is_validated": validated,
        "events": [],
        "mappings": {
            "field_correspondence": {
                "movement_id": "movement_id_src",
                "mode": "mode_src",
            },
            "value_correspondence": {
                "mode": {
                    "micro": "bus",
                    "subte": "metro",
                }
            },
        },
        "domains_effective": copy.deepcopy(schema_effective.domains_effective),
        "temporal": {"tier": "tier_3"},
    }

    if include_dataset_id:
        metadata["dataset_id"] = "dset_test_001"
    if include_artifact_id:
        metadata["artifact_id"] = "art_test_001"

    provenance = {
        "source": {"name": "synthetic", "entity": "trips"},
        "ingestion": {"created_at_utc": "2026-04-04T00:00:00Z"},
    }

    return TripDataset(
        data=make_trip_df(),
        schema=schema,
        schema_version=schema.version,
        provenance=provenance,
        field_correspondence={"movement_id": "movement_id_src", "mode": "mode_src"},
        value_correspondence={"mode": {"micro": "bus", "subte": "metro"}},
        metadata=metadata,
        schema_effective=schema_effective,
    )


def make_write_options(
    *,
    storage_format: str = "parquet",
    mode: str = "error_if_exists",
    require_validated: bool = True,
    parquet_compression: str | None = "snappy",
    feather_compression: str | None = "lz4",
    normalize_artifact_dir: bool = True,
) -> WriteTripsOptions:
    return WriteTripsOptions(
        mode=mode,
        require_validated=require_validated,
        storage_format=storage_format,
        parquet_compression=parquet_compression,
        feather_compression=feather_compression,
        normalize_artifact_dir=normalize_artifact_dir,
    )


def make_sidecar_payload(
    *,
    schema: TripSchema | None = None,
    schema_effective: TripSchemaEffective | None = None,
    metadata: dict | None = None,
    provenance: dict | None = None,
    dataset_id: str = "dset_sidecar_001",
    artifact_id: str = "art_sidecar_001",
    storage_format: str = "parquet",
    parquet_compression: str | None = "snappy",
    feather_compression: str | None = "lz4",
) -> dict:
    schema = schema or make_trip_schema_minimal()
    schema_effective = schema_effective or make_trip_schema_effective_minimal()

    options = make_write_options(
        storage_format=storage_format,
        parquet_compression=parquet_compression,
        feather_compression=feather_compression,
    )

    data_filename = _trip_data_filename_for_storage(storage_format)

    metadata = copy.deepcopy(metadata) if metadata is not None else {
        "dataset_id": dataset_id,
        "artifact_id": artifact_id,
        "is_validated": True,
        "events": [],
        "mappings": {
            "field_correspondence": {"movement_id": "movement_id_src"},
            "value_correspondence": {"mode": {"micro": "bus"}},
        },
        "domains_effective": copy.deepcopy(schema_effective.domains_effective),
    }

    provenance = copy.deepcopy(provenance) if provenance is not None else {
        "source": {"name": "synthetic", "entity": "trips"},
        "ingestion": {"created_at_utc": "2026-04-04T00:00:00Z"},
    }

    return {
        "dataset_type": "trips",
        "format": "golondrina",
        "layout_version": "1.1",
        "storage": {
            "format": storage_format,
            "options": _build_storage_options_snapshot(options),
        },
        "dataset_id": dataset_id,
        "artifact_id": artifact_id,
        "files": {
            "data": data_filename,
            "metadata": "trips.metadata.json",
        },
        "schema": _trip_schema_to_snapshot(schema),
        "schema_effective": _trip_schema_effective_to_snapshot(schema_effective),
        "provenance": provenance,
        "metadata": metadata,
    }

## Bloque 2. Helpers generales de layout, parámetros y JSON

Qué prueba: resolución de rutas, normalización del bundle `.golondrina`, nombres de archivos por backend, serialización de parámetros, eventos y JSON-safe.

### Test 2.1 - sufijo `.golondrina` y normalización del root

Qué prueba: que el helper detecte y agregue el sufijo canónico solo cuando corresponde.

In [7]:
p1 = Path("artifact")
p2 = Path("artifact.golondrina")

assert _has_golondrina_artifact_suffix(p1) is False
assert _has_golondrina_artifact_suffix(p2) is True

assert _append_golondrina_artifact_suffix(p1) == Path("artifact.golondrina")
assert _append_golondrina_artifact_suffix(p2) == Path("artifact.golondrina")

assert _normalize_trips_artifact_root_for_write(
    "my_bundle",
    normalize_artifact_dir=True,
) == Path("my_bundle.golondrina")

assert _normalize_trips_artifact_root_for_write(
    "my_bundle",
    normalize_artifact_dir=False,
) == Path("my_bundle")

show_ok("Test 2.1 - sufijo y normalización de root")

OK - Test 2.1 - sufijo y normalización de root


### Test 2.2 - `_trip_data_filename_for_storage`

Qué prueba: nombre canónico del archivo tabular según backend.

In [8]:
assert _trip_data_filename_for_storage("parquet") == "trips.parquet"
assert _trip_data_filename_for_storage("feather") == "trips.feather"

try:
    _trip_data_filename_for_storage("csv")
    raise AssertionError("Debió fallar por backend no soportado")
except ValueError:
    pass

show_ok("Test 2.2 - _trip_data_filename_for_storage")

OK - Test 2.2 - _trip_data_filename_for_storage


### Test 2.3 - `_build_storage_options_snapshot`

Qué prueba: snapshot persistible de opciones de storage para Parquet y Feather.

In [9]:
parquet_options = WriteTripsOptions(
    storage_format="parquet",
    parquet_compression="snappy",
    feather_compression="lz4",
)

feather_options = WriteTripsOptions(
    storage_format="feather",
    parquet_compression="snappy",
    feather_compression="lz4",
)

parquet_snapshot = _build_storage_options_snapshot(parquet_options)
feather_snapshot = _build_storage_options_snapshot(feather_options)

assert parquet_snapshot == {"compression": "snappy"}
assert feather_snapshot == {"compression": "lz4", "version": 2}

assert_json_safe(parquet_snapshot, "parquet_storage_snapshot")
assert_json_safe(feather_snapshot, "feather_storage_snapshot")

show_ok("Test 2.3 - _build_storage_options_snapshot")

OK - Test 2.3 - _build_storage_options_snapshot


### Test 2.4 - `_resolve_trips_artifact_paths`

Qué prueba: que el layout formal de trips resuelva root, sidecar oficial y sidecar legacy.

Nota: en la implementación actual el helper ya no expone `data_path`, porque el archivo tabular depende de `storage_format`.

In [10]:
root = Path("tmp_demo_artifact")
paths = _resolve_trips_artifact_paths(root)

assert paths.root_dir == root
assert paths.sidecar_path == root / "trips.metadata.json"
assert paths.legacy_sidecar_path == root / "metadata.json"

assert not hasattr(paths, "data_path"), "El layout vigente no debe asumir un data_path fijo"

show_ok("Test 2.4 - _resolve_trips_artifact_paths")

OK - Test 2.4 - _resolve_trips_artifact_paths


### Test 2.5 - `_options_to_write_parameters`

Qué prueba: serialización estable de parámetros efectivos de escritura.

In [11]:
params = _options_to_write_parameters(
    path=Path("artifact.golondrina"),
    options=WriteTripsOptions(
        mode="overwrite",
        require_validated=False,
        storage_format="feather",
        parquet_compression="snappy",
        feather_compression="zstd",
        normalize_artifact_dir=False,
    ),
)

assert params["path"] == str(Path("artifact.golondrina").expanduser())
assert params["mode"] == "overwrite"
assert params["require_validated"] is False
assert params["storage_format"] == "feather"
assert params["parquet_compression"] == "snappy"
assert params["feather_compression"] == "zstd"
assert params["normalize_artifact_dir"] is False

assert_json_safe(params, "write_parameters")

show_ok("Test 2.5 - _options_to_write_parameters")

OK - Test 2.5 - _options_to_write_parameters


### Test 2.6 - `_assert_json_safe`

Qué prueba: acepta payloads serializables y aborta ante objetos no serializables para sidecar.

In [12]:
issues = []
_assert_json_safe({"ok": [1, 2, 3]}, label="payload_ok", issues=issues)
assert issues == []

issues_bad = []
try:
    _assert_json_safe({"bad": {1, 2, 3}}, label="payload_bad", issues=issues_bad)
    raise AssertionError("Debió fallar por objeto no JSON-safe")
except ExportError:
    assert_issue_present(issues_bad, "WRT.JSON.NOT_SERIALIZABLE")

show_ok("Test 2.6 - _assert_json_safe")

OK - Test 2.6 - _assert_json_safe


### Test 2.7 - `_build_issues_summary`, `_build_io_event` y `_append_event`

Qué prueba: forma mínima de trazabilidad operacional usada por el evento `write_trips`.

In [13]:
issues = [
    Issue(level="info", code="CODE.INFO", message="info"),
    Issue(level="warning", code="CODE.WARN", message="warning"),
    Issue(level="warning", code="CODE.WARN", message="warning again"),
]

issues_summary = _build_issues_summary(issues)

assert issues_summary["counts"]["info"] == 1
assert issues_summary["counts"]["warning"] == 2
assert issues_summary["counts"]["error"] == 0
assert issues_summary["top_codes"][0]["code"] == "CODE.WARN"
assert issues_summary["top_codes"][0]["count"] == 2

event = _build_io_event(
    op="write_trips",
    parameters={"storage_format": "parquet"},
    summary={"n_rows": 3},
    issues_summary=issues_summary,
)

assert event["op"] == "write_trips"
assert "ts_utc" in event
assert event["parameters"]["storage_format"] == "parquet"
assert event["summary"]["n_rows"] == 3
assert event["issues_summary"]["counts"]["warning"] == 2

metadata_in = {"events": [{"op": "previous"}]}
metadata_out = _append_event(metadata_in, event)

assert len(metadata_in["events"]) == 1
assert len(metadata_out["events"]) == 2
assert metadata_out["events"][-1]["op"] == "write_trips"

assert_json_safe(event, "write_event")
assert_json_safe(metadata_out, "metadata_with_event")

show_ok("Test 2.7 - issues_summary + io_event + append_event")

OK - Test 2.7 - issues_summary + io_event + append_event


## Bloque 3. Preparación Arrow y escritura tabular backend-aware

Qué prueba: helpers nuevos que reemplazan la lógica antigua Parquet-only. En la implementación vigente, la preparación de categóricos se hace una sola vez para backends Arrow y luego se escribe como Parquet o Feather.

### Test 3.1 - `_collect_arrow_categorical_fields`

Qué prueba: detección de campos categóricos desde schema, `schema_effective.dtype_effective` y `schema_effective.domains_effective`.

In [14]:
df = pd.DataFrame(
    {
        "mode": ["bus", "metro"],
        "purpose": ["work", "study"],
        "gender": ["M", "F"],
        "comment": ["a", "b"],
    }
)

schema = make_trip_schema(
    [
        make_field("mode", "categorical"),
        make_field("purpose", "string"),
        make_field("gender", "string"),
        make_field("comment", "string"),
    ]
)

schema_effective = make_trip_schema_effective(
    dtype_effective={"purpose": "categorical"},
    domains_effective={"gender": {"values": ["M", "F"]}},
)

categorical_fields = _collect_arrow_categorical_fields(df, schema, schema_effective)

assert set(categorical_fields) == {"mode", "purpose", "gender"}
assert "comment" not in categorical_fields

show_ok("Test 3.1 - _collect_arrow_categorical_fields")

OK - Test 3.1 - _collect_arrow_categorical_fields


### Test 3.2 - `_prepare_trips_df_for_arrow_write`

Qué prueba: conversión a `pandas.Categorical` de los campos categóricos efectivos, eliminación de categorías no usadas y no-mutación del dataframe original.

In [15]:
df = pd.DataFrame(
    {
        "mode": pd.Series(
            pd.Categorical(["bus", "bus", "metro"], categories=["bus", "metro", "car"])
        ),
        "purpose": ["work", "study", "work"],
        "comment": ["ok1", "ok2", "ok3"],
        "trip_weight": [1.0, 2.0, 3.0],
    }
)

schema = make_trip_schema(
    [
        make_field("mode", "categorical"),
        make_field("purpose", "string"),
        make_field("comment", "string"),
        make_field("trip_weight", "float"),
    ]
)

schema_effective = make_trip_schema_effective(
    dtype_effective={"purpose": "categorical"},
    domains_effective={"purpose": {"values": ["work", "study", "health"]}},
)

prepared = _prepare_trips_df_for_arrow_write(df, schema, schema_effective)

assert isinstance(prepared["mode"].dtype, pd.CategoricalDtype)
assert list(prepared["mode"].cat.categories) == ["bus", "metro"]

assert isinstance(prepared["purpose"].dtype, pd.CategoricalDtype)
assert set(prepared["purpose"].cat.categories) == {"work", "study"}

assert not isinstance(prepared["comment"].dtype, pd.CategoricalDtype)
assert prepared["comment"].dtype == df["comment"].dtype
assert prepared["trip_weight"].dtype == df["trip_weight"].dtype

# No debe mutar el dataframe original
assert isinstance(df["mode"].dtype, pd.CategoricalDtype)
assert list(df["mode"].cat.categories) == ["bus", "metro", "car"]
assert df["purpose"].dtype == object

show_ok("Test 3.2 - _prepare_trips_df_for_arrow_write")

OK - Test 3.2 - _prepare_trips_df_for_arrow_write


### Test 3.3 - `_write_trips_table_to_staging` con Parquet

Qué prueba: escritura tabular Parquet desde el helper de OP-06, usando compresión efectiva y preservando columnas. También verifica a nivel Parquet que las columnas categóricas se materializan con dictionary encoding.

In [16]:
df = make_trip_df()
schema = make_trip_schema_minimal()
schema_effective = make_trip_schema_effective_minimal()

with tempfile.TemporaryDirectory() as td:
    root = Path(td) / "artifact"
    root.mkdir(parents=True, exist_ok=True)

    data_path = root / _trip_data_filename_for_storage("parquet")

    issues = []
    _write_trips_table_to_staging(
        df,
        data_path,
        storage_format="parquet",
        parquet_compression="snappy",
        feather_compression="lz4",
        schema=schema,
        schema_effective=schema_effective,
        issues=issues,
        destination_path=root,
    )

    assert data_path.exists()
    assert issues == []

    df_back = pd.read_parquet(data_path, engine="pyarrow")
    assert len(df_back) == len(df)
    assert list(df_back.columns) == list(df.columns)

    parquet_file = pq.ParquetFile(data_path)
    try:
        names = parquet_file.schema_arrow.names

        for col_name in ["mode", "purpose"]:
            col_idx = names.index(col_name)
            encodings = {
                str(enc).upper()
                for enc in parquet_file.metadata.row_group(0).column(col_idx).encodings
            }
            assert any("DICTIONARY" in enc for enc in encodings), (
                f"{col_name} no quedó con dictionary encoding: {encodings}"
            )
    finally:
        parquet_file.close()

show_ok("Test 3.3 - _write_trips_table_to_staging parquet")

OK - Test 3.3 - _write_trips_table_to_staging parquet


In [17]:
df = make_trip_df()
schema = make_trip_schema_minimal()
schema_effective = make_trip_schema_effective_minimal()

with tempfile.TemporaryDirectory() as td:
    root = Path(td) / "artifact"
    root.mkdir(parents=True, exist_ok=True)

    data_path = root / _trip_data_filename_for_storage("parquet")

    issues = []
    _write_trips_table_to_staging(
        df,
        data_path,
        storage_format="parquet",
        parquet_compression="snappy",
        feather_compression="lz4",
        schema=schema,
        schema_effective=schema_effective,
        issues=issues,
        destination_path=root,
    )

    assert data_path.exists()
    assert issues == []

    df_back = pd.read_parquet(data_path, engine="pyarrow")
    assert len(df_back) == len(df)
    assert list(df_back.columns) == list(df.columns)

    parquet_file = pq.ParquetFile(data_path)
    try:
        names = parquet_file.schema_arrow.names

        for col_name in ["mode", "purpose"]:
            col_idx = names.index(col_name)
            encodings = {
                str(enc).upper()
                for enc in parquet_file.metadata.row_group(0).column(col_idx).encodings
            }
            assert any("DICTIONARY" in enc for enc in encodings), (
                f"{col_name} no quedó con dictionary encoding: {encodings}"
            )
    finally:
        parquet_file.close()

show_ok("Test 3.3 - _write_trips_table_to_staging parquet")

OK - Test 3.3 - _write_trips_table_to_staging parquet


### Test 3.4 - `_write_trips_table_to_staging` con Feather

Qué prueba: escritura tabular Feather v2 desde el helper de OP-06, usando compresión efectiva y archivo `trips.feather`.

In [18]:
df = make_trip_df()
schema = make_trip_schema_minimal()
schema_effective = make_trip_schema_effective_minimal()

with tempfile.TemporaryDirectory() as td:
    root = Path(td) / "artifact"
    root.mkdir(parents=True, exist_ok=True)

    data_path = root / _trip_data_filename_for_storage("feather")

    issues = []
    _write_trips_table_to_staging(
        df,
        data_path,
        storage_format="feather",
        parquet_compression="snappy",
        feather_compression="lz4",
        schema=schema,
        schema_effective=schema_effective,
        issues=issues,
        destination_path=root,
    )

    assert data_path.exists()
    assert issues == []

    df_back = feather.read_feather(data_path)
    assert len(df_back) == len(df)
    assert list(df_back.columns) == list(df.columns)

    table = feather.read_table(data_path)
    assert table.schema.names == list(df.columns)

    for col_name in ["mode", "purpose"]:
        arrow_type = table.schema.field(col_name).type
        assert patypes.is_dictionary(arrow_type), (
            f"{col_name} no quedó como dictionary en Feather: {arrow_type}"
        )

show_ok("Test 3.4 - _write_trips_table_to_staging feather")

OK - Test 3.4 - _write_trips_table_to_staging feather


### Test 3.5 - `_write_trips_table_to_staging` falla con backend no soportado

Qué prueba: si llega un `storage_format` inválido a bajo nivel, el helper emite issue de escritura y aborta.

In [19]:
df = make_trip_df()
schema = make_trip_schema_minimal()
schema_effective = make_trip_schema_effective_minimal()

with tempfile.TemporaryDirectory() as td:
    root = Path(td) / "artifact"
    root.mkdir(parents=True, exist_ok=True)

    issues = []
    try:
        _write_trips_table_to_staging(
            df,
            root / "trips.unknown",
            storage_format="unknown",
            parquet_compression="snappy",
            feather_compression="lz4",
            schema=schema,
            schema_effective=schema_effective,
            issues=issues,
            destination_path=root,
        )
        raise AssertionError("Debió fallar por backend no soportado")
    except ExportError:
        # El helper clasifica cualquier formato distinto de feather como falla Parquet.
        assert_issue_present(issues, "WRT.PARQUET.WRITE_FAILED")

show_ok("Test 3.5 - _write_trips_table_to_staging backend inválido")

OK - Test 3.5 - _write_trips_table_to_staging backend inválido


## Bloque 4. Prechecks de contrato de escritura

Qué prueba: validaciones previas de OP-06 antes de tocar disco: tipo de input, superficie tabular, opciones, precondición de validación, destino y serialización JSON-safe.

### Test 4.1 - `_validate_write_contract` happy path

Qué prueba: caso correcto con `TripDataset` validado y opciones por defecto.

In [20]:
trips = make_tripdataset(validated=True)

with tempfile.TemporaryDirectory() as td:
    issues = []
    _validate_write_contract(
        trips,
        Path(td) / "artifact",
        WriteTripsOptions(),
        issues=issues,
    )

    assert_issue_absent(issues, "WRT.VALIDATION.REQUIRED_NOT_VALIDATED")
    assert_issue_absent(issues, "WRT.CORE.INVALID_TRIPDATASET")
    assert_issue_absent(issues, "WRT.CORE.INVALID_DATA_SURFACE")
    assert_issue_absent(issues, "WRT.OPTIONS.UNSUPPORTED_STORAGE_FORMAT")

show_ok("Test 4.1 - _validate_write_contract happy path")

OK - Test 4.1 - _validate_write_contract happy path


### Test 4.2 - `_validate_write_contract` fatal por dataset no validado

Qué prueba: `require_validated=True` exige `metadata["is_validated"] == True`.

In [21]:
trips = make_tripdataset(validated=False)

with tempfile.TemporaryDirectory() as td:
    issues = []
    try:
        _validate_write_contract(
            trips,
            Path(td) / "artifact",
            WriteTripsOptions(require_validated=True),
            issues=issues,
        )
        raise AssertionError("Debió fallar por dataset no validado")
    except ValidationError:
        assert_issue_present(issues, "WRT.VALIDATION.REQUIRED_NOT_VALIDATED")

show_ok("Test 4.2 - _validate_write_contract require_validated")

OK - Test 4.2 - _validate_write_contract require_validated


### Test 4.3 - `_validate_write_contract` permite dataset vacío con issue informativo

Qué prueba: un dataframe vacío es persistible, pero deja evidencia `WRT.CORE.EMPTY_DATAFRAME`.

In [22]:
trips = make_tripdataset(validated=True)
trips.data = trips.data.iloc[0:0].copy()

with tempfile.TemporaryDirectory() as td:
    issues = []
    _validate_write_contract(
        trips,
        Path(td) / "artifact",
        WriteTripsOptions(),
        issues=issues,
    )

    assert_issue_present(issues, "WRT.CORE.EMPTY_DATAFRAME")
    assert_counts_by_level(issues, info=1)

show_ok("Test 4.3 - _validate_write_contract empty dataframe")

OK - Test 4.3 - _validate_write_contract empty dataframe


### Test 4.4 - `_validate_write_contract` fatal por `storage_format` no soportado

Qué prueba: la operación solo acepta `parquet` y `feather`.

In [23]:
trips = make_tripdataset(validated=True)

with tempfile.TemporaryDirectory() as td:
    issues = []
    try:
        _validate_write_contract(
            trips,
            Path(td) / "artifact",
            WriteTripsOptions(storage_format="csv"),
            issues=issues,
        )
        raise AssertionError("Debió fallar por storage_format no soportado")
    except ExportError:
        assert_issue_present(issues, "WRT.OPTIONS.UNSUPPORTED_STORAGE_FORMAT")

show_ok("Test 4.4 - _validate_write_contract storage_format inválido")

OK - Test 4.4 - _validate_write_contract storage_format inválido


### Test 4.5 - `_validate_write_contract` fatal por compresión Parquet no soportada

Qué prueba: la compresión se valida según el backend efectivo.

In [24]:
trips = make_tripdataset(validated=True)

with tempfile.TemporaryDirectory() as td:
    issues = []
    try:
        _validate_write_contract(
            trips,
            Path(td) / "artifact",
            WriteTripsOptions(
                storage_format="parquet",
                parquet_compression="invalid_codec",
            ),
            issues=issues,
        )
        raise AssertionError("Debió fallar por compresión Parquet no soportada")
    except ExportError:
        assert_issue_present(issues, "WRT.OPTIONS.UNSUPPORTED_PARQUET_COMPRESSION")

show_ok("Test 4.5 - _validate_write_contract compresión parquet inválida")

OK - Test 4.5 - _validate_write_contract compresión parquet inválida


### Test 4.6 - `_validate_write_contract` fatal por compresión Feather no soportada

Qué prueba: Feather acepta solo las compresiones soportadas por el contrato vigente.

In [25]:
trips = make_tripdataset(validated=True)

with tempfile.TemporaryDirectory() as td:
    issues = []
    try:
        _validate_write_contract(
            trips,
            Path(td) / "artifact",
            WriteTripsOptions(
                storage_format="feather",
                feather_compression="gzip",
            ),
            issues=issues,
        )
        raise AssertionError("Debió fallar por compresión Feather no soportada")
    except ExportError:
        assert_issue_present(issues, "WRT.OPTIONS.UNSUPPORTED_FEATHER_COMPRESSION")

show_ok("Test 4.6 - _validate_write_contract compresión feather inválida")

OK - Test 4.6 - _validate_write_contract compresión feather inválida


### Test 4.7 - `_validate_write_contract` fatal por destino existente

Qué prueba: `mode="error_if_exists"` aborta si el root formal ya existe.

In [26]:
trips = make_tripdataset(validated=True)

with tempfile.TemporaryDirectory() as td:
    root = Path(td) / "artifact"
    root.mkdir()
    (root / "old.txt").write_text("legacy", encoding="utf-8")

    issues = []
    try:
        _validate_write_contract(
            trips,
            root,
            WriteTripsOptions(mode="error_if_exists", normalize_artifact_dir=False),
            issues=issues,
        )
        raise AssertionError("Debió fallar por destino existente")
    except ExportError:
        assert_issue_present(issues, "WRT.DEST.ALREADY_EXISTS")

show_ok("Test 4.7 - _validate_write_contract destino existente")

OK - Test 4.7 - _validate_write_contract destino existente


### Test 4.8 - `_validate_write_contract` fatal por metadata no JSON-safe

Qué prueba: los bloques que irán al sidecar deben ser serializables a JSON.

In [27]:
trips = make_tripdataset(validated=True)
trips.metadata["bad_object"] = {1, 2, 3}

with tempfile.TemporaryDirectory() as td:
    issues = []
    try:
        _validate_write_contract(
            trips,
            Path(td) / "artifact",
            WriteTripsOptions(),
            issues=issues,
        )
        raise AssertionError("Debió fallar por metadata no serializable")
    except ExportError:
        assert_issue_present(issues, "WRT.JSON.NOT_SERIALIZABLE")

show_ok("Test 4.8 - _validate_write_contract metadata no JSON-safe")

OK - Test 4.8 - _validate_write_contract metadata no JSON-safe


## Bloque 5. Identidad, sidecar y metadata persistible

Qué prueba: resolución de `dataset_id`, generación de `artifact_id`, construcción de sidecar backend-aware y evento `write_trips` antes de persistir.

### Test 5.1 - `_resolve_write_identity_and_sidecar` preserva `dataset_id`

Qué prueba: si `dataset_id` ya existe y es válido, se preserva; `artifact_id` se genera siempre nuevo.

In [28]:
trips = make_tripdataset(validated=True, include_dataset_id=True, include_artifact_id=False)
paths = _resolve_trips_artifact_paths(Path("tmp_artifact_for_resolve"))

resolved = _resolve_write_identity_and_sidecar(
    trips,
    paths,
    WriteTripsOptions(storage_format="parquet"),
    existing_issues=[],
)

assert resolved.dataset_id_status == "preserved"
assert resolved.dataset_id == "dset_test_001"
assert isinstance(resolved.artifact_id, str) and resolved.artifact_id.startswith("art_")

assert resolved.metadata_for_persist["dataset_id"] == resolved.dataset_id
assert resolved.metadata_for_persist["artifact_id"] == resolved.artifact_id
assert resolved.metadata_for_persist["is_validated"] is True
assert resolved.metadata_for_persist["events"][-1]["op"] == "write_trips"

assert resolved.sidecar_payload["dataset_id"] == resolved.dataset_id
assert resolved.sidecar_payload["artifact_id"] == resolved.artifact_id
assert resolved.sidecar_payload["files"]["data"] == "trips.parquet"
assert resolved.sidecar_payload["storage"]["format"] == "parquet"

assert resolved.issues == []
assert_json_safe(resolved.sidecar_payload, "sidecar_payload")

show_ok("Test 5.1 - _resolve_write_identity_and_sidecar preserve dataset_id")

OK - Test 5.1 - _resolve_write_identity_and_sidecar preserve dataset_id


### Test 5.2 - `_resolve_write_identity_and_sidecar` crea `dataset_id` faltante

Qué prueba: si falta identidad lógica, OP-06 la crea y deja issue informativo.

In [29]:
trips = make_tripdataset(validated=True, include_dataset_id=False, include_artifact_id=False)
paths = _resolve_trips_artifact_paths(Path("tmp_artifact_for_resolve"))

resolved = _resolve_write_identity_and_sidecar(
    trips,
    paths,
    WriteTripsOptions(storage_format="parquet"),
    existing_issues=[],
)

assert resolved.dataset_id_status == "created"
assert isinstance(resolved.dataset_id, str) and resolved.dataset_id.startswith("dset_")
assert isinstance(resolved.artifact_id, str) and resolved.artifact_id.startswith("art_")

assert_issue_present(resolved.issues, "WRT.METADATA.DATASET_ID_CREATED")
assert resolved.metadata_for_persist["events"][-1]["op"] == "write_trips"

show_ok("Test 5.2 - _resolve_write_identity_and_sidecar dataset_id creado")

OK - Test 5.2 - _resolve_write_identity_and_sidecar dataset_id creado


### Test 5.3 - `_resolve_write_identity_and_sidecar` regenera `dataset_id` inválido

Qué prueba: si `dataset_id` existe pero es vacío/no interpretable, OP-06 lo regenera con warning.

In [30]:
trips = make_tripdataset(validated=True, include_dataset_id=True, include_artifact_id=False)
trips.metadata["dataset_id"] = ""

paths = _resolve_trips_artifact_paths(Path("tmp_artifact_for_resolve"))

resolved = _resolve_write_identity_and_sidecar(
    trips,
    paths,
    WriteTripsOptions(storage_format="parquet"),
    existing_issues=[],
)

assert resolved.dataset_id_status == "regenerated"
assert isinstance(resolved.dataset_id, str) and resolved.dataset_id.startswith("dset_")
assert_issue_present(resolved.issues, "WRT.METADATA.DATASET_ID_REGENERATED")

show_ok("Test 5.3 - _resolve_write_identity_and_sidecar dataset_id regenerado")

OK - Test 5.3 - _resolve_write_identity_and_sidecar dataset_id regenerado


### Test 5.4 - sidecar backend-aware para Feather

Qué prueba: cuando `storage_format="feather"`, el sidecar apunta a `trips.feather`, registra `storage.format="feather"` y conserva versión Feather v2.

In [31]:
trips = make_tripdataset(validated=True)
paths = _resolve_trips_artifact_paths(Path("tmp_artifact_feather"))

resolved = _resolve_write_identity_and_sidecar(
    trips,
    paths,
    WriteTripsOptions(
        storage_format="feather",
        feather_compression="lz4",
    ),
    existing_issues=[],
)

sidecar = resolved.sidecar_payload

assert sidecar["storage"]["format"] == "feather"
assert sidecar["storage"]["options"]["compression"] == "lz4"
assert sidecar["storage"]["options"]["version"] == 2
assert sidecar["files"]["data"] == "trips.feather"
assert resolved.files_written == ["trips.feather", "trips.metadata.json"]

assert_json_safe(sidecar, "feather_sidecar")

show_ok("Test 5.4 - sidecar backend-aware feather")

OK - Test 5.4 - sidecar backend-aware feather


### Test 5.5 - evento `write_trips` incorpora issues previos

Qué prueba: el evento que se incrusta en metadata resume issues acumulados antes de construir el sidecar.

In [32]:
trips = make_tripdataset(validated=True, include_dataset_id=False)
paths = _resolve_trips_artifact_paths(Path("tmp_artifact_with_prior_issue"))

prior_issues = [
    Issue(
        level="info",
        code="CUSTOM.PRIOR.INFO",
        message="issue previo de prueba",
    )
]

resolved = _resolve_write_identity_and_sidecar(
    trips,
    paths,
    WriteTripsOptions(storage_format="parquet"),
    existing_issues=prior_issues,
)

event = resolved.metadata_for_persist["events"][-1]

assert event["op"] == "write_trips"
assert event["issues_summary"]["counts"]["info"] >= 1

codes_in_event = [x["code"] for x in event["issues_summary"]["top_codes"]]
assert "CUSTOM.PRIOR.INFO" in codes_in_event or "WRT.METADATA.DATASET_ID_CREATED" in codes_in_event

assert_json_safe(event, "write_event_with_prior_issues")

show_ok("Test 5.5 - evento write_trips resume issues")

OK - Test 5.5 - evento write_trips resume issues


### Test 5.6 - `_write_sidecar_json`

Qué prueba: escritura física del sidecar oficial `trips.metadata.json` sin usar helpers de lectura de OP-07.

In [33]:
with tempfile.TemporaryDirectory() as td:
    root = Path(td) / "artifact"
    root.mkdir(parents=True, exist_ok=True)
    paths = _resolve_trips_artifact_paths(root)

    payload = make_sidecar_payload(storage_format="feather", feather_compression="lz4")
    issues = []

    _write_sidecar_json(
        payload,
        paths.sidecar_path,
        issues=issues,
        destination_path=root,
        dataset_id=payload["dataset_id"],
        artifact_id=payload["artifact_id"],
    )

    assert paths.sidecar_path.exists()
    assert issues == []

    loaded = json.loads(paths.sidecar_path.read_text(encoding="utf-8"))
    assert loaded["dataset_type"] == "trips"
    assert loaded["format"] == "golondrina"
    assert loaded["storage"]["format"] == "feather"
    assert loaded["files"]["data"] == "trips.feather"
    assert loaded["files"]["metadata"] == "trips.metadata.json"

show_ok("Test 5.6 - _write_sidecar_json")

OK - Test 5.6 - _write_sidecar_json


### Test 5.7 - `_build_write_trips_summary`

Qué prueba: summary mínimo estable de OP-06.

In [34]:
summary = _build_write_trips_summary(
    n_rows=3,
    path=Path("/tmp/artifact"),
    artifact_id="art_001",
    dataset_id_status="created",
    dataset_id="dset_001",
    storage_format="feather",
    files_written=["trips.feather", "trips.metadata.json"],
)

assert summary["n_rows"] == 3
assert Path(summary["path"]) == Path("/tmp/artifact")
assert summary["artifact_id"] == "art_001"
assert summary["dataset_id"] == "dset_001"
assert summary["dataset_id_status"] == "created"
assert summary["storage_format"] == "feather"
assert summary["files_written"] == ["trips.feather", "trips.metadata.json"]

assert_json_safe(summary, "write_summary")

show_ok("Test 5.7 - _build_write_trips_summary")

OK - Test 5.7 - _build_write_trips_summary


## Bloque 6. Staging, completitud y commit final

Qué prueba: materialización segura en staging, verificación de completitud mínima, commit hacia destino formal, política de colisión y cleanup best-effort.

### Test 6.1 - `_create_trips_staging_dir` y `_cleanup_staging_dir`

Qué prueba: creación de staging hermano del destino y cleanup normal.

In [35]:
with tempfile.TemporaryDirectory() as td:
    final_dir = Path(td) / "artifact_final"
    issues = []

    staging_dir = _create_trips_staging_dir(final_dir, issues=issues)

    assert staging_dir.exists()
    assert staging_dir.is_dir()
    assert staging_dir.parent == final_dir.parent
    assert final_dir.name in staging_dir.name
    assert issues == []

    _cleanup_staging_dir(
        staging_dir,
        final_dir,
        ["trips.parquet", "trips.metadata.json"],
        issues,
    )

    assert not staging_dir.exists()

show_ok("Test 6.1 - staging create + cleanup")

OK - Test 6.1 - staging create + cleanup


### Test 6.2 - `_assert_staging_complete` para Parquet

Qué prueba: staging completo con `trips.parquet` y `trips.metadata.json`.

In [36]:
with tempfile.TemporaryDirectory() as td:
    staging_root = Path(td) / "staging_ok"
    staging_root.mkdir()

    paths = _resolve_trips_artifact_paths(staging_root)
    (staging_root / "trips.parquet").touch()
    paths.sidecar_path.touch()

    issues = []
    _assert_staging_complete(
        paths,
        expected_files=["trips.parquet", "trips.metadata.json"],
        issues=issues,
        destination_path=staging_root,
    )

    assert issues == []

show_ok("Test 6.2 - _assert_staging_complete parquet")

OK - Test 6.2 - _assert_staging_complete parquet


### Test 6.3 - `_assert_staging_complete` para Feather

Qué prueba: staging completo con `trips.feather` y `trips.metadata.json`.

In [37]:
with tempfile.TemporaryDirectory() as td:
    staging_root = Path(td) / "staging_ok"
    staging_root.mkdir()

    paths = _resolve_trips_artifact_paths(staging_root)
    (staging_root / "trips.feather").touch()
    paths.sidecar_path.touch()

    issues = []
    _assert_staging_complete(
        paths,
        expected_files=["trips.feather", "trips.metadata.json"],
        issues=issues,
        destination_path=staging_root,
    )

    assert issues == []

show_ok("Test 6.3 - _assert_staging_complete feather")

OK - Test 6.3 - _assert_staging_complete feather


### Test 6.4 - `_assert_staging_complete` fatal si falta artefacto requerido

Qué prueba: no se debe exponer un bundle incompleto como persistencia formal válida.

In [38]:
with tempfile.TemporaryDirectory() as td:
    staging_root = Path(td) / "staging_bad"
    staging_root.mkdir()

    paths = _resolve_trips_artifact_paths(staging_root)
    (staging_root / "trips.parquet").touch()

    issues = []
    try:
        _assert_staging_complete(
            paths,
            expected_files=["trips.parquet", "trips.metadata.json"],
            issues=issues,
            destination_path=staging_root,
        )
        raise AssertionError("Debió fallar por staging incompleto")
    except ExportError:
        assert_issue_present(issues, "WRT.IO.STAGING_INCOMPLETE")

show_ok("Test 6.4 - _assert_staging_complete incompleto")

OK - Test 6.4 - _assert_staging_complete incompleto


### Test 6.5 - `_commit_staged_trips_artifact` success

Qué prueba: commit correcto desde staging al directorio final cuando el destino no existe.

In [39]:
with tempfile.TemporaryDirectory() as td:
    parent = Path(td)
    staging = parent / "staging"
    final_dir = parent / "artifact"

    staging.mkdir()
    (staging / "trips.parquet").write_text("dummy parquet placeholder", encoding="utf-8")
    (staging / "trips.metadata.json").write_text("{}", encoding="utf-8")

    issues = []
    _commit_staged_trips_artifact(
        staging,
        final_dir,
        mode="error_if_exists",
        files_written=["trips.parquet", "trips.metadata.json"],
        issues=issues,
    )

    assert final_dir.exists()
    assert not staging.exists()
    assert (final_dir / "trips.parquet").exists()
    assert (final_dir / "trips.metadata.json").exists()
    assert issues == []

show_ok("Test 6.5 - _commit_staged_trips_artifact success")

OK - Test 6.5 - _commit_staged_trips_artifact success


### Test 6.6 - `_commit_staged_trips_artifact` overwrite

Qué prueba: `mode="overwrite"` reemplaza el destino existente.

In [40]:
with tempfile.TemporaryDirectory() as td:
    parent = Path(td)
    final_dir = parent / "artifact"
    staging = parent / "staging"

    final_dir.mkdir()
    (final_dir / "old.txt").write_text("legacy", encoding="utf-8")

    staging.mkdir()
    (staging / "trips.feather").write_text("dummy feather placeholder", encoding="utf-8")
    (staging / "trips.metadata.json").write_text("{}", encoding="utf-8")

    issues = []
    _commit_staged_trips_artifact(
        staging,
        final_dir,
        mode="overwrite",
        files_written=["trips.feather", "trips.metadata.json"],
        issues=issues,
    )

    assert final_dir.exists()
    assert not staging.exists()
    assert not (final_dir / "old.txt").exists()
    assert (final_dir / "trips.feather").exists()
    assert (final_dir / "trips.metadata.json").exists()
    assert issues == []

show_ok("Test 6.6 - _commit_staged_trips_artifact overwrite")

OK - Test 6.6 - _commit_staged_trips_artifact overwrite


### Test 6.7 - `_commit_staged_trips_artifact` fatal por colisión

Qué prueba: `mode="error_if_exists"` aborta si el destino formal ya existe.

In [41]:
with tempfile.TemporaryDirectory() as td:
    parent = Path(td)
    final_dir = parent / "artifact"
    staging = parent / "staging"

    final_dir.mkdir()
    (final_dir / "old.txt").write_text("legacy", encoding="utf-8")

    staging.mkdir()
    (staging / "trips.parquet").write_text("dummy parquet placeholder", encoding="utf-8")
    (staging / "trips.metadata.json").write_text("{}", encoding="utf-8")

    issues = []
    try:
        _commit_staged_trips_artifact(
            staging,
            final_dir,
            mode="error_if_exists",
            files_written=["trips.parquet", "trips.metadata.json"],
            issues=issues,
        )
        raise AssertionError("Debió fallar por destino ya existente")
    except ExportError:
        assert_issue_present(issues, "WRT.DEST.ALREADY_EXISTS")

show_ok("Test 6.7 - _commit_staged_trips_artifact collision")

OK - Test 6.7 - _commit_staged_trips_artifact collision
